# Build the Firm-Year Panel for the Privacy Patent Regression

| Item | Description |
|---|---|
| Goal | One row per analysis-sample firm and fiscal year with the number of privacy patents filed in the year, the firm's topic-relevance score (one score per firm, as in T4.1.3), its NAICS two-digit code, and the three lagged controls (log firm age, log total assets, log R&D expense at t − 1), ready for the Poisson regression of T4.1.6. |
| Input | `output/X1.1.3/analysis_sample_firms.csv` (the 689 sample firms with at least one patent filed in 1999 to 2017, with their relevance score); `output/T3.4.1/filings.csv` from T3 (the sample firm-years); `Processed/privacy_patents.parquet` from T4.1.2 and `Processed/sample_firm_patents.parquet` from T4.1.1; `crsp_compustat_link.dta` in the section's `Raw` folder on scratch; `compustat_controls.csv` (Compustat annual fundamentals: `gvkey`, `fyear`, `at`, `xrd`, with format flags) in the section's `compustat` folder in the project tree, so that the regression inputs sit next to the notebooks. |
| Output | On scratch, `Processed/privacy_regression_panel.dta` and `.csv`. |
| Unit | A firm (gvkey) and fiscal year with a 10-K in the all-years T3 sample, fiscal `YEAR_START` to `YEAR_END` (1999 to 2021), for the firms of the T4.1.3 analysis sample. |
| Variables | • `n_privacy_patents`: $Y_{it} = \sum_{p \in i} \mathbb{1}[\text{filing year}_p = t]\, P_p$, the number of KPSS patents firm $i$ filed in calendar year $t$ that carry a code of the privacy list (T4.1.2)<br>• `n_patents_all`: the same count over all of the firm's KPSS patents, kept for reference<br>• `relevance_firm`: $R_i$, the cosine similarity between a firm-level summary of firm $i$'s 10-K Item 1 sections and the policy summary of legislator statements on consumer data privacy legislation (T3.8.3, `summaries \| policy_summary \| firm_summary`), one score per firm, constant over its years<br>• `naics2`: the first two digits of the firm's Compustat NAICS code, for the sector-by-year fixed effects<br>• `ln_age_l1`: $\ln(\text{age}_{i,t-1})$, age in years at $t-1$ since the firm's first CRSP listing, at least one<br>• `ln_at_l1`: $\ln(\text{AT}_{i,t-1})$, log total assets at $t-1$, Compustat `at`, positive values only<br>• `ln_xrd_l1`: $\ln(1 + \text{XRD}_{i,t-1})$, log of one plus R&D expense at $t-1$, Compustat `xrd`, with missing set to zero<br>• `rd_missing_l1`: $\mathbb{1}[\text{XRD}_{i,t-1} \text{ missing}]$ |
| Timing | The count of fiscal year $t$ is the count filed in calendar year $t$, exact for December fiscal year-ends and off by up to six months otherwise. Controls are the Compustat values of fiscal year $t-1$. The relevance score does not vary over time. |
| Fixed effects | Because the score is constant within firm, a firm fixed effect would absorb it exactly. T4.1.6 therefore absorbs NAICS two-digit-by-year cells and clusters standard errors by firm. |

See `T4.1.6` for the regression, `T4.1.3` for the analysis sample and the score, and `T4.1.2` for the privacy patents.


## Step 1. Import packages


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


## Step 2. Define file locations and the period

The two roots and the S5 folders; the T3 filing list is read from T3's `output` folder, the link file from the section's `Raw` folder on scratch, and the Compustat controls from the section's `compustat` folder in the project tree.


In [2]:
PROJECT_FOLDER = Path("/project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis")
SCRATCH_FOLDER = Path("/scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis")
NOTEBOOK_SECTION = Path("X1 - Example of Research Enabled by LLM Pipeline")
NOTEBOOK_SUBFOLDER = NOTEBOOK_SECTION / "X1.1 - Privacy Patents and Topic Relevance"
NOTEBOOK_NAME = "T4.1.5"  # the identifier names this notebook's output folder
RAW_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SECTION / "Raw"  # section level, as in T4.1.1
PROCESSED_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Processed"
OUTPUT_FOLDER = PROJECT_FOLDER / NOTEBOOK_SECTION / "output" / NOTEBOOK_NAME
T3_OUTPUT = PROJECT_FOLDER / "T3 - Topic Relevance and Corporate Disclosure/TEMP/output"
ANALYSIS_FIRMS_FILE = PROJECT_FOLDER / NOTEBOOK_SECTION / "output/X1.1.3/analysis_sample_firms.csv"
FILINGS_FILE = T3_OUTPUT / "T3.4.1/filings.csv"
PRIVACY_PATENTS_FILE = PROCESSED_FOLDER / "privacy_patents.parquet"  # T4.1.2
PATENTS_FILE = PROCESSED_FOLDER / "sample_firm_patents.parquet"  # T4.1.1
LINK_FILE = RAW_FOLDER / "crsp_compustat_link.dta"
CONTROLS_FILE = PROJECT_FOLDER / NOTEBOOK_SECTION / "compustat/compustat_controls.csv"  # copied from Raw so that the section folder holds the regression inputs

SAMPLE = "all_years"  # the T3 sample the filings come from
YEAR_START = 1999  # fiscal years kept; later filings are not fully granted in the KPSS data
YEAR_END = 2021
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
print(f"Processed: {PROCESSED_FOLDER}")
print(f"Compustat controls: {CONTROLS_FILE}")
assert CONTROLS_FILE.exists(), f"missing {CONTROLS_FILE}"


Processed: /scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis/T4 - Toy Example [Topic Relevance on Privacy Innovation]/T4.1 - Privacy Patents and Topic Relevance/Processed
Compustat controls: /project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis/T4 - Toy Example [Topic Relevance on Privacy Innovation]/compustat/compustat_controls.csv


## Step 3. Start from the analysis firms' 10-K years and attach the firm-level relevance score

The T3 filing list gives one row per firm and fiscal year; it is kept for the analysis-sample firms and the period. The firm's score from T4.1.3 is joined by firm and repeats across its years.


In [3]:
analysis_firms = pd.read_csv(ANALYSIS_FIRMS_FILE, dtype={"gvkey": str})
filings = pd.read_csv(FILINGS_FILE, dtype={"gvkey": str, "cik": str})
panel = filings.loc[filings[f"in_{SAMPLE}"].eq(1) & filings["gvkey"].isin(analysis_firms["gvkey"]) & filings["fyear"].between(YEAR_START, YEAR_END),
                    ["gvkey", "conm", "fyear", "accessionNo"]].rename(columns={"accessionNo": "accession_no"}).reset_index(drop=True)
panel = panel.merge(analysis_firms[["gvkey", "relevance_score"]].rename(columns={"relevance_score": "relevance_firm"}), on="gvkey", how="left", validate="many_to_one")
assert not panel.duplicated(["gvkey", "fyear"]).any(), "a firm with two filings in one fiscal year"
assert panel["relevance_firm"].notna().all(), "a firm without a relevance score"
firms_without_filing = analysis_firms.loc[~analysis_firms["gvkey"].isin(panel["gvkey"]), "conm"].tolist()
print(f"Firm-years: {len(panel):,}; firms: {panel['gvkey'].nunique():,} of the {len(analysis_firms):,} analysis firms; fiscal years {panel['fyear'].min()} to {panel['fyear'].max()}")
print(f"Analysis firms without a 10-K in the period, left out: {len(firms_without_filing)} ({', '.join(firms_without_filing)})")
display(panel.head(3))


Firm-years: 11,464; firms: 685 of the 689 analysis firms; fiscal years 1999 to 2021
Analysis firms without a 10-K in the period, left out: 4 (CRANE CO, BLACKROCK INC, BUNGE GLOBAL SA, NOBLE CORP PLC)


,gvkey,conm,fyear,accession_no,relevance_firm
0,001013,ADC TELECOMMUNICATIONS INC,1999,0000912057-99-011120,0.522551
1,001013,ADC TELECOMMUNICATIONS INC,2000,0000912057-01-000926,0.522551
2,001013,ADC TELECOMMUNICATIONS INC,2001,0000912057-02-001614,0.522551


## Step 4. Count each firm's privacy patents and all patents by filing year

The privacy patents of T4.1.2 and all sample-firm patents of T4.1.1, counted by firm and filing year and joined on the fiscal year; firm-years without a patent get zero.


In [4]:
privacy_patents = pd.read_parquet(PRIVACY_PATENTS_FILE, columns=["patent_num", "gvkey", "filing_year"])
patents = pd.read_parquet(PATENTS_FILE, columns=["patent_num", "gvkey", "filing_year"])
privacy_counts = privacy_patents.groupby(["gvkey", "filing_year"]).size().rename("n_privacy_patents")
all_counts = patents.groupby(["gvkey", "filing_year"]).size().rename("n_patents_all")
panel = panel.merge(privacy_counts, left_on=["gvkey", "fyear"], right_index=True, how="left").merge(all_counts, left_on=["gvkey", "fyear"], right_index=True, how="left")
panel[["n_privacy_patents", "n_patents_all"]] = panel[["n_privacy_patents", "n_patents_all"]].fillna(0).astype(int)
print(f"Firm-years with a privacy patent: {int(panel['n_privacy_patents'].gt(0).sum()):,} of {len(panel):,} ({panel['n_privacy_patents'].gt(0).mean():.1%}); "
      f"firms with a privacy patent in some year: {panel.loc[panel['n_privacy_patents'].gt(0), 'gvkey'].nunique():,}; privacy patents in the panel: {int(panel['n_privacy_patents'].sum()):,}; "
      f"all patents: {int(panel['n_patents_all'].sum()):,}")


Firm-years with a privacy patent: 2,015 of 11,464 (17.6%); firms with a privacy patent in some year: 265; privacy patents in the panel: 32,252; all patents: 1,160,525


## Step 5. Attach the NAICS two-digit code and the first CRSP listing year

Both come from the CRSP-Compustat link table: the first two digits of the firm's Compustat NAICS code, and the year of the earliest link date across the firm's links, which starts the firm-age clock.


In [5]:
link = pd.read_stata(LINK_FILE, columns=["GVKEY", "naics", "LINKDT"])
link["gvkey"] = link["GVKEY"].astype(str).str.zfill(6)
link = link.loc[link["gvkey"].isin(panel["gvkey"])]
link["naics"] = link["naics"].astype(str).str.strip().replace({"": None, "nan": None})
naics = link.dropna(subset=["naics"])
assert naics.groupby("gvkey")["naics"].nunique().le(1).all(), "a gvkey with two NAICS codes"
firms = naics.drop_duplicates("gvkey").set_index("gvkey")[["naics"]]
firms["naics2"] = firms["naics"].str[:2]
firms["first_listing_year"] = link.groupby("gvkey")["LINKDT"].min().dt.year
panel = panel.merge(firms, left_on="gvkey", right_index=True, how="left", validate="many_to_one")
assert panel[["naics2", "first_listing_year"]].notna().all().all(), "a firm without a NAICS code or a listing year"
print(f"Firms: {len(firms):,}; two-digit NAICS codes: {firms['naics2'].nunique()}; first listing year {int(firms['first_listing_year'].min())} to {int(firms['first_listing_year'].max())}")
display(firms["naics2"].value_counts().rename("firms").to_frame().T)


Firms: 685; two-digit NAICS codes: 23; first listing year 1949 to 2017


naics2,33,32,51,52,31,22,21,54,44,48,...,72,53,62,99,23,49,71,61,81,11
firms,210,101,79,69,38,38,34,22,14,13,...,9,9,5,3,3,2,2,2,1,1


## Step 6. Attach the lagged Compustat controls

Compustat annual fundamentals, industrial format in US dollars, one row per firm and fiscal year; the values of fiscal year t − 1 are joined to the row of fiscal year t. Log total assets needs positive assets. R&D expense is missing for most firm-years, so its log takes one plus the expense with missing set to zero, and an indicator keeps the information. Firm age at t − 1 is the years since the first listing year, at least one.


In [6]:
controls = pd.read_csv(CONTROLS_FILE, dtype={"gvkey": str, "naics": str})
controls = controls.loc[controls["indfmt"].eq("INDL") & controls["curcd"].eq("USD") & controls["gvkey"].isin(panel["gvkey"]), ["gvkey", "fyear", "at", "xrd"]]
assert not controls.duplicated(["gvkey", "fyear"]).any(), "duplicate Compustat firm-years among the analysis firms"
lagged = controls.rename(columns={"fyear": "lag_fyear", "at": "at_l1", "xrd": "xrd_l1"})
panel["lag_fyear"] = panel["fyear"] - 1
panel = panel.merge(lagged, on=["gvkey", "lag_fyear"], how="left", validate="one_to_one").drop(columns="lag_fyear")
panel["ln_at_l1"] = np.log(panel["at_l1"].where(panel["at_l1"] > 0))
panel["rd_missing_l1"] = panel["xrd_l1"].isna().astype(int)
panel["ln_xrd_l1"] = np.log1p(panel["xrd_l1"].fillna(0).clip(lower=0))
panel["age_l1"] = (panel["fyear"] - 1 - panel["first_listing_year"] + 1).clip(lower=1)
panel["ln_age_l1"] = np.log(panel["age_l1"])
print(f"Firm-years with a Compustat row at t − 1: {int(panel['at_l1'].notna().sum()):,} of {len(panel):,}; with positive assets: {int(panel['ln_at_l1'].notna().sum()):,}; "
      f"with reported R&D: {int((panel['rd_missing_l1'] == 0).sum()):,}")
display(panel[["gvkey", "fyear", "n_privacy_patents", "relevance_firm", "naics2", "age_l1", "ln_age_l1", "at_l1", "ln_at_l1", "xrd_l1", "ln_xrd_l1", "rd_missing_l1"]].head(3))


Firm-years with a Compustat row at t − 1: 11,406 of 11,464; with positive assets: 11,406; with reported R&D: 7,334


,gvkey,fyear,n_privacy_patents,relevance_firm,naics2,age_l1,ln_age_l1,at_l1,ln_at_l1,xrd_l1,ln_xrd_l1,rd_missing_l1
0,001013,1999,1,0.522551,33,20,2.995732,1300.587,7.170571,137.912,4.933841,0
1,001013,2000,1,0.522551,33,21,3.044522,1672.529,7.422092,281.472,5.643579,0
2,001013,2001,0,0.522551,33,22,3.091042,3970.500,8.286647,338.000,5.826000,0


## Step 7. Check the panel, label the variables, and save it

One row per firm and fiscal year, counts non-negative, every row with a NAICS code and a score. Saved as Stata and CSV for T4.1.6; the regression sample (rows with all three controls observed) is counted here as a preview.


In [7]:
assert not panel.duplicated(["gvkey", "fyear"]).any()
assert (panel[["n_privacy_patents", "n_patents_all"]] >= 0).all().all()
panel = panel[["gvkey", "conm", "fyear", "accession_no", "naics", "naics2", "n_privacy_patents", "n_patents_all", "relevance_firm",
               "first_listing_year", "age_l1", "ln_age_l1", "at_l1", "ln_at_l1", "xrd_l1", "ln_xrd_l1", "rd_missing_l1"]].sort_values(["gvkey", "fyear"]).reset_index(drop=True)
labels = {
    "gvkey": "Compustat firm identifier", "conm": "Compustat firm name", "fyear": "Fiscal year", "accession_no": "10-K accession number (T3 sample)",
    "naics": "Compustat NAICS code", "naics2": "NAICS two-digit code",
    "n_privacy_patents": "Privacy patents filed in the year (T4.1.2 list)", "n_patents_all": "All patents filed in the year",
    "relevance_firm": "Cosine: firm-level Item 1 summary vs policy summary (one per firm)",
    "first_listing_year": "Year of first CRSP link (listing)", "age_l1": "Years since first listing, t-1", "ln_age_l1": "Log firm age, t-1",
    "at_l1": "Total assets, t-1 (millions USD)", "ln_at_l1": "Log total assets, t-1", "xrd_l1": "R&D expense, t-1 (millions USD)",
    "ln_xrd_l1": "Log(1 + R&D expense), t-1, missing as 0", "rd_missing_l1": "R&D expense missing at t-1 (1 = yes)",
}
panel.to_stata(PROCESSED_FOLDER / "privacy_regression_panel.dta", write_index=False, version=118, variable_labels=labels)
panel.to_csv(PROCESSED_FOLDER / "privacy_regression_panel.csv", index=False, float_format="%.6f")
regression_rows = panel.dropna(subset=["ln_age_l1", "ln_at_l1", "ln_xrd_l1"])
print(f"Saved: {PROCESSED_FOLDER / 'privacy_regression_panel.dta'} and .csv ({len(panel):,} firm-years, {panel['gvkey'].nunique():,} firms, {len(panel.columns)} variables)")
print(f"Rows with all three lagged controls: {len(regression_rows):,} firm-years, {regression_rows['gvkey'].nunique():,} firms; "
      f"of these, firms with a privacy patent in some year: {regression_rows.loc[regression_rows['n_privacy_patents'].gt(0), 'gvkey'].nunique():,}")
display(panel[["n_privacy_patents", "n_patents_all", "relevance_firm", "ln_age_l1", "ln_at_l1", "ln_xrd_l1", "rd_missing_l1"]].describe().round(3))


Saved: /scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis/T4 - Toy Example [Topic Relevance on Privacy Innovation]/T4.1 - Privacy Patents and Topic Relevance/Processed/privacy_regression_panel.dta and .csv (11,464 firm-years, 685 firms, 17 variables)
Rows with all three lagged controls: 11,406 firm-years, 685 firms; of these, firms with a privacy patent in some year: 265


,n_privacy_patents,n_patents_all,relevance_firm,ln_age_l1,ln_at_l1,ln_xrd_l1,rd_missing_l1
count,11464.000,11464.000,11464.000,11464.000,11406.000,11464.000,11464.00
mean,2.813,101.232,0.516,3.111,9.000,3.023,0.36
std,17.806,389.694,0.035,0.943,1.712,2.844,0.48
min,0.000,0.000,0.406,0.000,2.067,0.000,0.00
25%,0.000,0.000,0.490,2.639,7.874,0.000,0.00
50%,0.000,5.000,0.514,3.332,8.890,3.489,0.00
75%,0.000,48.000,0.540,3.850,10.072,5.405,1.00
max,586.000,9868.000,0.621,4.277,15.198,10.663,1.00
